In [ ]:
import numpy as np
import pandas as pd

from utils import constants, metrics, io_results, visualization, data
from utils.client import VotingProcessor
from utils.voting import VotingProbabilities, VotingPick, PARTY_NAMES
from SoD.data_processing_utils import decline_region_to_Genitiv, decline_region_to_locative


## Experiment Setup

In [ ]:
MODEL = constants.GPT_41_NANO
# MODEL = constants.GPT_4o_MINI
# MODEL = constants.GPT_41
# MODEL = constants.GPT_4o

processor = VotingProcessor()

In [ ]:
def citizen_prompt(region):
    return "Jsem obyvatel/ka "+ decline_region_to_Genitiv(region) + ". Ve volbách do poslanecké sněmovny v roce 2021 jsem volil:"
def direct_prompt(region):
    return "Účast a výsledek voleb do poslanecké sněmovny v "+ decline_region_to_locative(region)+ " v roce 2021 budou:"

In [5]:
print(citizen_prompt(constants.COUNTRY))
print(direct_prompt(constants.COUNTRY))

Jsem obyvatel/ka Česka. Ve volbách do poslanecké sněmovny v roce 2021 jsem volil:
Účast a výsledek voleb do poslanecké sněmovny v Česku v roce 2021 budou:


## Election Preferences Experiment

In [6]:
n = 10
prompt = citizen_prompt
# prompt = direct_prompt

In [ ]:
latent_results = []
for region in constants.REGIONS_ALL:
    voting_results, skipped = processor.run(
        data=pd.DataFrame(np.random.rand(n, 1), columns=['Col']),
        prompt_creator= lambda res: prompt(region),
        response_model= VotingProbabilities,
        model= MODEL, temperature = 0,
    )
    results_df = io_results.results_to_dataframe(voting_results)
    model_latent_result = data.aggregate_results_from_df(results_df)
    model_latent_result['region'] = region
    latent_results.append(model_latent_result)

latent_results_df = pd.DataFrame(latent_results)
latent_results_df.to_csv(f"results/LatentPreferences_{MODEL}_{prompt.__name__}.csv")

## Visualization

In [ ]:
latent_results_df = pd.read_csv(f"results/LatentPreferences_{MODEL}_{prompt.__name__}.csv")
actual_results_df = io_results.load_actual_results()

for region in constants.REGIONS_ALL:
     pred = latent_results_df[latent_results_df['region'] == region].iloc[0]
     actual = actual_results_df[actual_results_df["region"] == region].iloc[0]
     fig = visualization.visualize_comprehensive_results(
        pred,
        actual,
        constants.PARTY_COLUMNS_2021,
        model_name=MODEL + " " + region, actual_is_claimed=False
    )

In [ ]:
cit = pd.read_csv(f"results/LatentPreferences_{MODEL}_{citizen_prompt.__name__}.csv")
dir = pd.read_csv(f"results/LatentPreferences_{MODEL}_{direct_prompt.__name__}.csv")


for region in constants.All_REGIONS:
     pred = cit[cit['region'] == region].iloc[0]
     actual = dir[dir["region"] == region].iloc[0]
     fig = visualization.visualize_comprehensive_results(
        pred,
        actual,
        constants.PARTY_COLUMNS_2021,
        model_name=MODEL + " " + region, actual_is_claimed=False
    )